In [1]:
import numpy as np
print(np.__version__) # doit afficher 1.x ou 2.x, pas une erreur

2.0.2


Phase 1 : Neurone unique, forward pass et calcul d'erreur


In [2]:
import numpy as np

# Données d'entrée (4 points en 2D)
X = np.array([
    [0.2, 0.1],
    [0.8, 0.9],
    [0.3, 0.7],
    [0.9, 0.2],
])
y = np.array([0, 1, 1, 0])

def sigmoid(x):
    """Fonction d'activation pour transformer la sortie en probabilité (0 à 1)."""
    return 1 / (1 + np.exp(-x))

def forward(X, w, b):
    """Calcule la sortie du neurone : z = Xw + b, puis activation."""
    z = np.dot(X, w) + b
    return sigmoid(z)

def compute_loss(y_true, y_pred):
    """Calcule l'erreur via la Binary Cross-Entropy."""
    # On évite log(0) ou log(1) qui renvoient NaN ou Inf
    y_pred = np.clip(y_pred, 1e-7, 1 - 1e-7)
    term_0 = y_true * np.log(y_pred)
    term_1 = (1 - y_true) * np.log(1 - y_pred)
    return -np.mean(term_0 + term_1)

# --- TEST SCÉNARIO INITIAL ---
w = np.array([0.5, -0.3])
b = 0.1

y_pred = forward(X, w, b)
loss = compute_loss(y, y_pred)

print(f"--- Scénario Fixe ---")
print(f"Prédictions : {y_pred.round(3)}")
print(f"Étiquettes  : {y}")
print(f"Loss BCE     : {loss:.4f}")

# --- TEST SCÉNARIO ADVERSARIAL (Poids à zéro) ---
w_zero = np.zeros(2)
b_zero = 0
y_pred_zero = forward(X, w_zero, b_zero)
loss_zero = compute_loss(y, y_pred_zero)

print(f"\n--- Scénario Adversarial (Zéro) ---")
print(f"Prédictions : {y_pred_zero}")
print(f"Loss BCE     : {loss_zero:.4f} (Doit être proche de 0.693)")

--- Scénario Fixe ---
Prédictions : [0.542 0.557 0.51  0.62 ]
Étiquettes  : [0 1 1 0]
Loss BCE     : 0.7519

--- Scénario Adversarial (Zéro) ---
Prédictions : [0.5 0.5 0.5 0.5]
Loss BCE     : 0.6931 (Doit être proche de 0.693)


Phase 2 : Descente de gradient à la main, loss par epoch

In [6]:
import numpy as np
import matplotlib
matplotlib.use('Agg') # Pour générer l'image sans interface graphique
import matplotlib.pyplot as plt

# Données
X = np.array([[0.2, 0.1], [0.8, 0.9], [0.3, 0.7], [0.9, 0.2]])
y = np.array([0, 1, 1, 0])

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def compute_loss(y_true, y_pred):
    y_pred = np.clip(y_pred, 1e-7, 1 - 1e-7)
    return -np.mean(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))

# Initialisation
np.random.seed(42)
w = np.random.randn(2) * 0.01
b = 0.0
learning_rate = 0.1
n_epochs = 50
losses = []

print("Début de l'entraînement...")

for epoch in range(n_epochs):
    # --- FORWARD PASS ---
    z = np.dot(X, w) + b
    y_pred = sigmoid(z)

    # Calcul et stockage de la loss
    loss = compute_loss(y, y_pred)
    losses.append(loss)

    # --- BACKPROPAGATION (Gradient Descent) ---
    # L'erreur résiduelle
    error = y_pred - y

    # Calcul des dérivées partielles (Gradients)
    n = len(y)
    dw = np.dot(X.T, error) / n
    db = np.mean(error)

    # --- MISE À JOUR DES PARAMÈTRES ---
    w -= learning_rate * dw
    b -= learning_rate * db

    if epoch % 10 == 0:
        print(f"Epoch {epoch:3d} | Loss: {loss:.4f} | w: {w.round(3)} | b: {b:.3f}")

# Visualisation
plt.figure(figsize=(8, 4))
plt.plot(losses, label='Loss BCE')
plt.xlabel("Epoch")
plt.ylabel("Loss BCE")
plt.title(f"Convergence (LR={learning_rate})")
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend()
plt.savefig("phase2_loss_curve.png", dpi=100, bbox_inches='tight')

print(f"\nCourbe sauvegardée : phase2_loss_curve.png")
print(f"Loss finale : {losses[-1]:.4f}")

Début de l'entraînement...
Epoch   0 | Loss: 0.6934 | w: [0.005 0.015] | b: -0.000
Epoch  10 | Loss: 0.6688 | w: [-0.001  0.17 ] | b: -0.010
Epoch  20 | Loss: 0.6468 | w: [-0.014  0.316] | b: -0.032
Epoch  30 | Loss: 0.6265 | w: [-0.033  0.453] | b: -0.063
Epoch  40 | Loss: 0.6074 | w: [-0.054  0.585] | b: -0.098

Courbe sauvegardée : phase2_loss_curve.png
Loss finale : 0.5910


Phase 3 : XOR, échec du neurone unique, victoire de la couche
cachée



In [13]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# 1. Données XOR (Incontournables)
X_xor = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=float)
y_xor = np.array([0, 1, 1, 0])

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def compute_loss_bce(y_true, y_pred):
    y_pred = np.clip(y_pred, 1e-7, 1 - 1e-7)
    return -np.mean(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))

# 2. Initialisation 2-2-1 (C'est ici qu'on "fixe" le problème)
# On utilise une seed différente et une distribution uniforme plus large
np.random.seed(1)
W1 = np.random.uniform(low=-2.0, high=2.0, size=(2, 2))
b1 = np.zeros(2)
W2 = np.random.uniform(low=-2.0, high=2.0, size=(2, 1))
b2 = np.zeros(1)

# Paramètres musclés pour sortir du plateau
learning_rate = 1.0
n_epochs = 10001
losses = []

print("Entraînement du MLP XOR : Objectif 100%...")

for epoch in range(n_epochs):
    # --- FORWARD PASS ---
    z1 = np.dot(X_xor, W1) + b1
    a1 = sigmoid(z1)

    z2 = np.dot(a1, W2) + b2
    a2 = sigmoid(z2)
    y_pred = a2.flatten()

    # Loss
    loss = compute_loss_bce(y_xor, y_pred)
    losses.append(loss)

    # --- BACKPROPAGATION ---
    n = len(y_xor)
    error2 = a2 - y_xor.reshape(-1, 1)

    # Gradients Sortie
    dW2 = np.dot(a1.T, error2) / n
    db2 = np.mean(error2, axis=0)

    # Gradients Couche Cachée (La clé du XOR)
    error1 = np.dot(error2, W2.T) * (a1 * (1 - a1))
    dW1 = np.dot(X_xor.T, error1) / n
    db1 = np.mean(error1, axis=0)

    # --- UPDATE ---
    W1 -= learning_rate * dW1
    b1 -= learning_rate * db1
    W2 -= learning_rate * dW2
    b2 -= learning_rate * db2

    # Affichage régulier
    if epoch % 2000 == 0:
        acc = np.mean((y_pred > 0.5) == y_xor)
        print(f"Epoch {epoch:5d} | Loss: {loss:.4f} | Accuracy: {acc:.2%}")

# --- ANALYSE FINALE ---
final_acc = np.mean((y_pred > 0.5) == y_xor)
print(f"\n--- RÉSULTAT FINAL ---")
print(f"Loss finale : {losses[-1]:.4f}")
print(f"Accuracy finale : {final_acc:.2%}")

if final_acc == 1.0:
    print("✅ Succès : Le XOR est résolu !")
else:
    print("❌ Échec : Toujours bloqué. Essayez de changer la seed.")

# --- PLOT DE LA FRONTIÈRE ---
xx, yy = np.meshgrid(np.linspace(-0.5, 1.5, 200), np.linspace(-0.5, 1.5, 200))
grid = np.c_[xx.ravel(), yy.ravel()]
z1g = sigmoid(np.dot(grid, W1) + b1)
z2g = sigmoid(np.dot(z1g, W2) + b2).reshape(xx.shape)

plt.figure(figsize=(8, 6))
plt.contourf(xx, yy, z2g, alpha=0.4, cmap='RdBu')
plt.scatter(X_xor[:, 0], X_xor[:, 1], c=y_xor, s=100, edgecolors='k', cmap='RdBu')
plt.title(f"XOR Résolu - Accuracy: {final_acc:.2%}")
plt.savefig("phase3_xor_success.png")

Entraînement du MLP XOR : Objectif 100%...
Epoch     0 | Loss: 0.8774 | Accuracy: 50.00%
Epoch  2000 | Loss: 0.0058 | Accuracy: 100.00%
Epoch  4000 | Loss: 0.0026 | Accuracy: 100.00%
Epoch  6000 | Loss: 0.0017 | Accuracy: 100.00%
Epoch  8000 | Loss: 0.0012 | Accuracy: 100.00%
Epoch 10000 | Loss: 0.0010 | Accuracy: 100.00%

--- RÉSULTAT FINAL ---
Loss finale : 0.0010
Accuracy finale : 100.00%
✅ Succès : Le XOR est résolu !


Phase 4 : Spirale 2D, frontière non-linéaire visualisée



In [16]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

def generate_spiral(n_points=400, noise=0.15, seed=42):
    np.random.seed(seed)
    n = n_points // 2
    theta = np.sqrt(np.random.rand(n)) * 4 * np.pi

    # Spirale 1
    X0 = np.c_[-np.cos(theta) * theta + np.random.randn(n) * noise,
               np.sin(theta) * theta + np.random.randn(n) * noise]
    # Spirale 2
    X1 = np.c_[np.cos(theta) * theta + np.random.randn(n) * noise,
               -np.sin(theta) * theta + np.random.randn(n) * noise]

    X = np.vstack([X0, X1]) / 15.0 # Normalisation pour aider la convergence
    y = np.hstack([np.zeros(n), np.ones(n)])
    return X, y

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def relu(x):
    return np.maximum(0, x)

def relu_grad(x):
    return (x > 0).astype(float)

def bce_loss(y_true, y_pred):
    y_pred = np.clip(y_pred, 1e-7, 1 - 1e-7)
    return -np.mean(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))

# 1. Préparation des données
X, y = generate_spiral(n_points=400, noise=0.15)
n_samples = len(y)

# 2. Initialisation He (Améliorée)
np.random.seed(42)
# On utilise 64 neurones par couche
W1 = np.random.randn(2, 64) * np.sqrt(2 / 2)
b1 = np.zeros(64)
W2 = np.random.randn(64, 64) * np.sqrt(2 / 64)
b2 = np.zeros(64)
W3 = np.random.randn(64, 1) * np.sqrt(2 / 64)
b3 = np.zeros(1)

# 3. Hyperparamètres ajustés
lr = 0.05 # Augmenté pour accélérer la convergence
n_epochs = 5000 # Augmenté pour laisser le temps de sculpter la spirale
losses = []

print("Entraînement Profond (2-64-64-1) - Objectif > 90%...")

for epoch in range(n_epochs):
    # --- FORWARD ---
    z1 = np.dot(X, W1) + b1
    a1 = relu(z1)

    z2 = np.dot(a1, W2) + b2
    a2 = relu(z2)

    z3 = np.dot(a2, W3) + b3
    y_pred_raw = sigmoid(z3)
    y_pred = y_pred_raw.flatten()

    # Loss
    loss = bce_loss(y, y_pred)
    losses.append(loss)

    # --- BACKWARD ---
    err3 = y_pred_raw - y.reshape(-1, 1)
    dW3 = np.dot(a2.T, err3) / n_samples
    db3 = np.mean(err3, axis=0)

    err2 = np.dot(err3, W3.T) * relu_grad(z2)
    dW2 = np.dot(a1.T, err2) / n_samples
    db2 = np.mean(err2, axis=0)

    err1 = np.dot(err2, W2.T) * relu_grad(z1)
    dW1 = np.dot(X.T, err1) / n_samples
    db1 = np.mean(err1, axis=0)

    # --- UPDATE ---
    W1 -= lr * dW1
    b1 -= lr * db1
    W2 -= lr * dW2
    b2 -= lr * db2
    W3 -= lr * dW3
    b3 -= lr * db3

    if epoch % 1000 == 0:
        acc = np.mean((y_pred > 0.5) == y)
        print(f"Epoch {epoch:4d} | Loss: {loss:.4f} | Accuracy: {acc:.2%}")

# --- RÉSULTATS ---
final_acc = np.mean((y_pred > 0.5) == y)
print(f"\nLoss finale : {losses[-1]:.4f}")
print(f"Accuracy finale : {final_acc:.2%}")

# --- PLOT FINAL ---
h = 0.02
xx, yy = np.meshgrid(np.arange(X[:, 0].min() - 0.1, X[:, 0].max() + 0.1, h),
                     np.arange(X[:, 1].min() - 0.1, X[:, 1].max() + 0.1, h))
grid = np.c_[xx.ravel(), yy.ravel()]
a1g = relu(np.dot(grid, W1) + b1)
a2g = relu(np.dot(a1g, W2) + b2)
zg = sigmoid(np.dot(a2g, W3) + b3).reshape(xx.shape)

plt.figure(figsize=(10, 8))
plt.contourf(xx, yy, zg, alpha=0.5, cmap='RdBu')
plt.scatter(X[:, 0], X[:, 1], c=y, cmap='RdBu', s=20, edgecolors='k')
plt.title(f"Spirale - Accuracy: {final_acc:.2%}")
plt.savefig("phase4_spirale_success.png")

Entraînement Profond (2-64-64-1) - Objectif > 90%...
Epoch    0 | Loss: 0.7261 | Accuracy: 50.00%
Epoch 1000 | Loss: 0.6007 | Accuracy: 67.25%
Epoch 2000 | Loss: 0.5852 | Accuracy: 59.00%
Epoch 3000 | Loss: 0.5705 | Accuracy: 57.25%
Epoch 4000 | Loss: 0.5462 | Accuracy: 60.25%

Loss finale : 0.4795
Accuracy finale : 72.50%


Phase 5 : Passage à Keras sur MNIST flatten
